# Data Preparation (Preparação dos dados)

### Biblioteca / Configuração

In [1]:
# Dependências
import sys
#!{sys.executable} -m pip install --disable-pip-version-check -r ../requirements.txt -q
print('Bibliotecas instaladas')

Bibliotecas instaladas


In [2]:
# Acesso aos modulos do diretório
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT)) 

# Manipulação dos dados
import pandas as pd
import numpy as np
import pickle

# Visualização dos dados
import matplotlib.pyplot as plt
import seaborn as sns

# Diretórios
from config.paths import *
from config.data_preprocessing import *

# Avisos
import warnings
warnings.filterwarnings('ignore')

# Configuração
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', None)

print('Ambiente Configurado')

Diretórios carregadas com sucesso
Funções extras carregadas com sucesso
Ambiente Configurado


## Parâmetros Globais

In [3]:
# define a coluna alvo do modelo
TARGET = 'FPD'
# percentual máximo de valores ausentes permitido para manter a variável
PERCENTUAL_MAX_FALTANTES = 70

### Carregamento dos dados 

In [4]:
# lê o arquivo parquet e carrega em um DataFrame
abt00 = pd.read_parquet(RAW_DIR / 'book_variaveis_04_v2.parquet')

## Tratamento inicial

### Grupo Controle

In [5]:
# gerar e salvar o grupo controle no data/processed

# cria flag para identificar clientes do grupo controle (CPF 6º e 7º dígitos = ZZ ou ZX)
abt00['FLAG_GRUPO_CONTROLE'] = (abt00['NUM_CPF'].astype(str).str[5:7].isin(['ZZ', 'ZX']).astype(int))

# filtra grupo controle
controle = abt00[abt00['FLAG_GRUPO_CONTROLE'] == 1]

# salva em parquet usando path centralizado
GRUPO_CONTROLE_FILE = RAW_DIR / 'grupo_controle.parquet'
controle.to_parquet(GRUPO_CONTROLE_FILE, index=False)

print(f"Grupo controle salvo em: {GRUPO_CONTROLE_FILE}")
print(f"Registros: {len(controle):,}")

Grupo controle salvo em: C:\Users\billy.reis\Desktop\hackathon_dev\01_data\raw\grupo_controle.parquet
Registros: 60,197


### Definir filtro grupo controle

In [6]:
# controla aplicação do filtro e remove coluna se só houver grupo controle
APLICAR_FILTRO_PADRAO = True  # True = sem grupo controle | False = base completa

if APLICAR_FILTRO_PADRAO:
    abt01 = abt00[abt00['FLAG_GRUPO_CONTROLE'] == 0].copy()

    # se depois do filtro só existir grupo controle, remove a coluna
    if 'FLAG_GRUPO_CONTROLE' in abt01.columns and abt01['FLAG_GRUPO_CONTROLE'].nunique() == 1:
        abt01.drop(columns=['FLAG_GRUPO_CONTROLE'], inplace=True)
else:
    abt01 = abt00.copy()

print(f"Modo ativo: {'Sem grupo controle' if APLICAR_FILTRO_PADRAO else 'base completa'}")
print(f"Base ativa: {len(abt01):,} registros")

Modo ativo: Sem grupo controle
Base ativa: 1,220,631 registros


### Separação dos dados para validação temporal (Out-of-Time)

A separação dos dados é realizada com base na **safra**, respeitando a ordem temporal das observações.  
Essa abordagem, conhecida como **validação Out-of-Time (OOT)**, evita vazamento de informação e simula o comportamento real do modelo em dados futuros.

In [7]:
# garante SAFRA como inteiro
abt01['SAFRA'] = abt01['SAFRA'].astype(int)
safra_counts = abt01['SAFRA'].value_counts().sort_index()

# Definir SAFRAs de teste (Fevereiro e Março 2025)
test_safras = [202502, 202503]

# Criar máscaras
test_mask = abt01['SAFRA'].isin(test_safras)
train_mask = ~test_mask

# Separar dados
train = abt01[train_mask].copy()
test = abt01[test_mask].copy()

train.shape, test.shape

((831081, 129), (389550, 129))

In [8]:
# salvar base de teste em parquet no diretório de predictions
TEST_FILE = RAW_DIR / 'base_test.parquet'
test.to_parquet(TEST_FILE, index=False)

print(f"Base de teste salva em: {TEST_FILE}")
print(f"Registros: {len(test):,}")

Base de teste salva em: C:\Users\billy.reis\Desktop\hackathon_dev\01_data\raw\base_test.parquet
Registros: 389,550


In [9]:
# Backup dos dados originais
train_01 = train.copy()

# lista de vars para retirar dos tratamentos e algumas colunas desnecessárias
ignore_cols = ['SAFRA', 'FPD', 'NUM_CPF', 'DATADENASCIMENTO', 'DATA_SAFRA', 'REGIAO_POSTAL_TXT', 'var_25']

# Aplicando no treino
train_01 = train_01.drop(columns=ignore_cols)

>Motivo da exclusão

- `REGIAO_POSTAL_TXT` replica o conteúdo de `REGIAO_POSTAL`, diferenciando-se apenas pelo tipo de dado.
- `var25` (categórica original) foi removida após sua binarização em variáveis dummies, evitando redundância e multicolinearidade no modelo.

In [10]:
# exibir header claro para geração da tabela de metadados do treino

print("METADADOS DO DATASET DE TREINO")
print('=' * 30)
print(f"Linhas: {train_01.shape[0]:,} | Colunas: {train_01.shape[1]:,}")
print('Gerando tabela de diagnóstico das variáveis...\n')

metadados = dataset_info_table(train_01)

print('OK: tabela de metadados criada com sucesso.')

METADADOS DO DATASET DE TREINO
Linhas: 831,081 | Colunas: 122
Gerando tabela de diagnóstico das variáveis...

OK: tabela de metadados criada com sucesso.


### Remoção de Colunas Desnecessárias

In [11]:
# filtra variáveis com muitos nulos e cardinalidade igual a 1
df_low_card = metadados[(metadados['PC_nulos'] >= PERCENTUAL_MAX_FALTANTES) | (metadados['Cardinalidade'] <= 1)]
df_low_card = list(df_low_card.Feature.values)

# efeito real do drop
qtd_excluir = train_01.columns.isin(df_low_card).sum()

print(qtd_excluir)
print(df_low_card)

3
['FLAG_INSTALACAO', 'PROD', 'flag_mig2']


In [12]:
# filtra variáveis com cardinalidade maior igual a 100000
df_high_card = metadados[(metadados['Cardinalidade'] >= 100000)]
df_high_card = list(df_high_card.Feature.values)

# efeito real do drop
qtd_excluir = train_01.columns.isin(df_high_card).sum()

print(qtd_excluir)
print(df_high_card)

1
['VAL_REAL_ULT_6_SAFRAS']


In [13]:
# Unir colunas de baixa e alta cardinalidade
drop_card = list(set(df_low_card + df_high_card))
# Remover colunas de baixa e alta cardinalidade
train_01 = train_01.drop(columns=drop_card, errors='ignore')

In [14]:
# colunas que permaneceram após o drop
features_pre_selection = train_01.columns.tolist()

artifact_path = Path(ARTIFACT_DIR) / 'features_pre_selection.pkl'

with open(artifact_path, 'wb') as f:
    pickle.dump(features_pre_selection, f)

print(f"Features pre seleção: {len(features_pre_selection)}")
print(f"Arquivo: {artifact_path}")

Features pre seleção: 118
Arquivo: C:\Users\billy.reis\Desktop\hackathon_dev\04_artifact\features_pre_selection.pkl


### Tratamento de Valores Faltantes

In [15]:
# Seleciona apenas colunas do tipo object/categoricas
cols_obj = train_01.select_dtypes(include='object').columns

# Identifica colunas que têm algum valor parecido com "desconhecido" (qualquer capitalização ou espaços)
cols_com_desconhecido = [
    col for col in cols_obj
    if train_01[col].astype(str).str.strip().str.match(r'(?i)^desconhecido$').any()
]
# Substituir todas as variações de "Desconhecido" por NaN
for col in cols_com_desconhecido:
    train_01[col] = train_01[col].astype(str).str.strip().replace(r'(?i)^desconhecido$', np.nan, regex=True)

In [16]:
# Seleciona colunas que não são float nem int
cols_nao_numericas = train_01.select_dtypes(exclude=['float', 'int']).columns.tolist()

# Converter para inteiro que aceita NaN
for col in cols_nao_numericas:
    train_01[col] = train_01[col].astype('Int64')  # 'Int64' com I maiúsculo é o tipo nullable integer do pandas

In [17]:
# Análise de missing values restantes
train_01, stats = custom_fillna(train_01)

In [18]:
# saneia SCORE_RATEO (inf/outliers) via cap [-10, 10] para manter robustez do pipeline
train_01['SCORE_RATEO'] = train_01['SCORE_RATEO'].clip(0, 10)

In [19]:
# salva estatísticas de imputação
artifact_path = Path(ARTIFACT_DIR) / 'stats_nulo.pkl'
with open(artifact_path, "wb") as f:
    pickle.dump(stats, f)

print(f"Colunas com imputação aplicada: {len(stats)}")
print(f"Artefato salvo em: {artifact_path}")

Colunas com imputação aplicada: 3
Artefato salvo em: C:\Users\billy.reis\Desktop\hackathon_dev\04_artifact\stats_nulo.pkl


## Salvamento dos Dados Processados

In [20]:
# Reanexar target Para treino
abt01_train_final = train_01.copy()
abt01_train_final[TARGET] = train[TARGET].values

In [21]:
# salvar base de Treino tratada
TRAIN_FILE = PROCESSED_DIR / 'abt01_train.parquet'
print('\n💾 Salvando dados processados...')

# salvar dataset final
abt01_train_final.to_parquet(TRAIN_FILE, index=False)
print(f"✓ Treino salvo: {TRAIN_FILE}")
print('✅ Persistência concluída')


💾 Salvando dados processados...
✓ Treino salvo: C:\Users\billy.reis\Desktop\hackathon_dev\01_data\processed\abt01_train.parquet
✅ Persistência concluída
